# 🌍 VORQ — Event Extractor Training (Colab)

Simple, error-free geopolitical event classification model training.

## What This Does
- Trains DistilBERT to classify scenarios into 10 event types
- Saves model locally (no Google Drive needed)
- Run all cells in order → Get trained model ready to download

## Time: ~2-3 hours on T4 GPU

After training:
1. Download `vorq_model.zip` from Colab files
2. Extract on Mac to `vorq/models/event_classifier/`
3. Restart VORQ server
4. Done! 🎉


In [1]:
# Install dependencies
import sys
!{sys.executable} -m pip install -q transformers datasets torch scikit-learn
print('✅ Ready to train')

✅ Ready to train



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Setup - Local Execution
import os
import json
import torch
import gc
from datetime import datetime

# Use local directory instead of /content/
MODEL_DIR = './vorq_model_output'
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'✅ Model will save to: {os.path.abspath(MODEL_DIR)}')
print(f'📅 Started: {datetime.now().strftime("%H:%M:%S")}\n')

✅ Model will save to: c:\Users\abhir\OneDrive\Desktop\vorq\vorq_model_output
📅 Started: 12:52:05



In [3]:
# Check GPU
print('System Check:')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
if torch.cuda.is_available():
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('✅ Ready!\n')


System Check:
  GPU: CPU only
✅ Ready!



In [4]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.8.0+cpu
CUDA Available: False


## 📊 Training Data

We create a synthetic training dataset with labeled geopolitical events.
Each sample has: `text` (scenario description) → `label` (event type)

In [5]:
# ── Training data: geopolitical event classification ──
import random

LABEL_MAP = {
    'war': 0, 'sanctions': 1, 'pandemic': 2, 'natural_disaster': 3,
    'supply_shock': 4, 'political_crisis': 5, 'economic_crisis': 6,
    'cyberattack': 7, 'energy_crisis': 8, 'trade_agreement': 9,
}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

# Synthetic training samples (expand this for production)
TRAINING_DATA = [
    # War
    ('Military conflict erupts between two major Asian powers', 'war'),
    ('Armed forces deploy to contested border region', 'war'),
    ('Naval confrontation in South China Sea escalates to armed conflict', 'war'),
    ('Air strikes reported on military installations', 'war'),
    ('Full-scale invasion launched across national border', 'war'),
    ('Missile attacks on civilian infrastructure in conflict zone', 'war'),
    ('Military buildup along disputed territory border', 'war'),
    ('Ground troops advance into neighboring territory', 'war'),
    ('War between India and China threatens global stability', 'war'),
    ('Russia launches military operation in Eastern Europe', 'war'),
    ('NATO allies prepare for potential military engagement', 'war'),
    ('Casualties mount as armed conflict enters second week', 'war'),
    ('China invades Taiwan sparking international crisis', 'war'),
    ('Military strike targets critical infrastructure', 'war'),
    ('Bombing campaign intensifies in regional conflict', 'war'),
    # Sanctions
    ('New economic sanctions imposed on technology exports', 'sanctions'),
    ('Trade embargo restricts semiconductor shipments', 'sanctions'),
    ('US bans chip exports to rival nations', 'sanctions'),
    ('Comprehensive sanctions package targets energy sector', 'sanctions'),
    ('Export controls tightened on advanced technology', 'sanctions'),
    ('Financial sanctions freeze assets of foreign entities', 'sanctions'),
    ('Trade restrictions expand to include rare earth minerals', 'sanctions'),
    ('Tariff increases on imported goods from major producer', 'sanctions'),
    ('US sanctions on semiconductor exports to China', 'sanctions'),
    ('EU imposes trade restrictions on Russian energy', 'sanctions'),
    ('Banking sector excluded from international SWIFT system', 'sanctions'),
    ('New tariff regime disrupts bilateral trade flows', 'sanctions'),
    ('Technology transfer ban affects chip manufacturers', 'sanctions'),
    ('Economic blockade impacts food and fuel imports', 'sanctions'),
    ('Export ban on critical defense components', 'sanctions'),
    # Pandemic
    ('New virus strain emerges with high transmissibility', 'pandemic'),
    ('WHO declares global pandemic emergency', 'pandemic'),
    ('Lockdown measures implemented across multiple countries', 'pandemic'),
    ('Respiratory disease outbreak overwhelms healthcare systems', 'pandemic'),
    ('Travel restrictions imposed as infection rates surge', 'pandemic'),
    ('Global pandemic similar to COVID-19 disrupts supply chains', 'pandemic'),
    ('Mass quarantine measures affect manufacturing output', 'pandemic'),
    ('Epidemic crosses international borders rapidly', 'pandemic'),
    ('Vaccine development race begins amid rising cases', 'pandemic'),
    ('Factory shutdowns due to workforce illness', 'pandemic'),
    ('Healthcare system collapse in multiple nations', 'pandemic'),
    ('Border closures halt international trade', 'pandemic'),
    ('New disease variant evades existing immunization', 'pandemic'),
    ('Global death toll rises as outbreak spreads', 'pandemic'),
    ('Schools and businesses close due to health emergency', 'pandemic'),
    # Natural disaster
    ('Major earthquake strikes densely populated region', 'natural_disaster'),
    ('Category 5 hurricane makes landfall', 'natural_disaster'),
    ('Tsunami warning issued after undersea seismic event', 'natural_disaster'),
    ('Devastating floods displace millions', 'natural_disaster'),
    ('Volcanic eruption disrupts air travel globally', 'natural_disaster'),
    ('Severe drought threatens agricultural output', 'natural_disaster'),
    ('Major earthquake hits Japan destroying industrial zones', 'natural_disaster'),
    ('Wildfire season causes unprecedented destruction', 'natural_disaster'),
    ('Cyclone damages port infrastructure', 'natural_disaster'),
    ('Landslides block major trade routes', 'natural_disaster'),
    ('Record flooding paralyzes transportation network', 'natural_disaster'),
    ('Earthquake and tsunami combination devastates coastline', 'natural_disaster'),
    ('Extreme weather event disrupts power grid', 'natural_disaster'),
    ('Glacial melt causes catastrophic river flooding', 'natural_disaster'),
    ('Tornado swarm destroys manufacturing facilities', 'natural_disaster'),
    # Supply shock
    ('Critical component shortage halts production lines', 'supply_shock'),
    ('Container ship blockage disrupts global shipping', 'supply_shock'),
    ('Semiconductor shortage affects auto manufacturing', 'supply_shock'),
    ('Raw material supply disruption from mining strike', 'supply_shock'),
    ('Port congestion causes major delivery delays', 'supply_shock'),
    ('Key supplier factory fire creates bottleneck', 'supply_shock'),
    ('Global chip shortage worsens amid high demand', 'supply_shock'),
    ('Logistics network collapse from driver shortage', 'supply_shock'),
    ('Rare earth supply cut threatens electronics production', 'supply_shock'),
    ('Major shipping lane closed due to security threat', 'supply_shock'),
    ('Supply chain breakdown cascades through industries', 'supply_shock'),
    ('Critical medicine shortage impacts healthcare', 'supply_shock'),
    ('Food supply disruption from export restrictions', 'supply_shock'),
    ('Steel shortage delays construction projects globally', 'supply_shock'),
    ('Warehouse capacity crisis during peak season', 'supply_shock'),
    # Political crisis
    ('Military coup overthrows elected government', 'political_crisis'),
    ('Mass protests destabilize ruling administration', 'political_crisis'),
    ('Constitutional crisis triggers political paralysis', 'political_crisis'),
    ('Opposition leader arrested amid political crackdown', 'political_crisis'),
    ('Disputed election results spark civil unrest', 'political_crisis'),
    ('Government collapse leads to power vacuum', 'political_crisis'),
    ('Regional separatist movement gains momentum', 'political_crisis'),
    ('Political assassination shocks international community', 'political_crisis'),
    ('Populist revolution challenges established order', 'political_crisis'),
    ('Democratic institutions undermined by authoritarian actions', 'political_crisis'),
    ('Mass civil disobedience paralyzes major cities', 'political_crisis'),
    ('Government shutdown over budget disagreement', 'political_crisis'),
    ('Ethnic tensions escalate into violent confrontations', 'political_crisis'),
    ('State of emergency declared after political turmoil', 'political_crisis'),
    ('International community condemns political repression', 'political_crisis'),
    # Economic crisis
    ('Stock market crashes wiping out trillions in value', 'economic_crisis'),
    ('Major bank defaults on international obligations', 'economic_crisis'),
    ('Currency collapse triggers hyperinflation', 'economic_crisis'),
    ('Sovereign debt crisis threatens eurozone stability', 'economic_crisis'),
    ('Housing market bubble bursts dramatically', 'economic_crisis'),
    ('Central bank emergency intervention fails to stem panic', 'economic_crisis'),
    ('US debt default crisis triggers global recession', 'economic_crisis'),
    ('Bond market selloff raises borrowing costs sharply', 'economic_crisis'),
    ('Multiple corporate bankruptcies signal systemic risk', 'economic_crisis'),
    ('Credit markets freeze as confidence evaporates', 'economic_crisis'),
    ('GDP contraction enters recession territory', 'economic_crisis'),
    ('Pension fund insolvency threatens retirement savings', 'economic_crisis'),
    ('Global financial contagion spreads from emerging markets', 'economic_crisis'),
    ('Inflation spiral erodes purchasing power rapidly', 'economic_crisis'),
    ('Banking system requires massive government bailout', 'economic_crisis'),
    # Cyberattack
    ('Massive ransomware attack cripples hospital systems', 'cyberattack'),
    ('State-sponsored hackers breach financial networks', 'cyberattack'),
    ('Critical infrastructure targeted by sophisticated malware', 'cyberattack'),
    ('Data breach exposes millions of customer records', 'cyberattack'),
    ('Power grid compromised by cyber intrusion', 'cyberattack'),
    ('Global banking system hit by coordinated cyberattack', 'cyberattack'),
    ('Supply chain attack through compromised software update', 'cyberattack'),
    ('Telecommunications network disrupted by DDoS attack', 'cyberattack'),
    ('Military systems breached by foreign hackers', 'cyberattack'),
    ('Online payment processing halted by security breach', 'cyberattack'),
    ('Critical government databases encrypted by ransomware', 'cyberattack'),
    ('Stock exchange trading halted after cyber incident', 'cyberattack'),
    ('Water treatment facility compromised remotely', 'cyberattack'),
    ('Major tech company suffers zero-day exploit', 'cyberattack'),
    ('Satellite communication systems disrupted by hack', 'cyberattack'),
    # Energy crisis
    ('Oil prices spike after OPEC announces production cuts', 'energy_crisis'),
    ('Natural gas shortage triggers heating emergency', 'energy_crisis'),
    ('Major pipeline explosion disrupts fuel supply', 'energy_crisis'),
    ('Oil supply shock in Middle East affects global markets', 'energy_crisis'),
    ('Refinery fire reduces gasoline production capacity', 'energy_crisis'),
    ('Nuclear power plant shutdown increases grid strain', 'energy_crisis'),
    ('LNG supply disruption from geopolitical tensions', 'energy_crisis'),
    ('Fuel shortage causes transport paralysis', 'energy_crisis'),
    ('Energy prices triple amid supply concerns', 'energy_crisis'),
    ('Grid failure leaves millions without electricity', 'energy_crisis'),
    ('Oil embargo triggers global energy crisis', 'energy_crisis'),
    ('Coal supply disruption impacts power generation', 'energy_crisis'),
    ('Renewable energy shortfall during weather extremes', 'energy_crisis'),
    ('Strategic petroleum reserve depletion concerns market', 'energy_crisis'),
    ('Gas pipeline dispute between producing nations', 'energy_crisis'),
    # Trade agreement
    ('Major trade deal signed reducing bilateral tariffs', 'trade_agreement'),
    ('Free trade agreement opens new market access', 'trade_agreement'),
    ('Regional trade partnership expands membership', 'trade_agreement'),
    ('Trade normalization agreement improves diplomatic relations', 'trade_agreement'),
    ('Economic cooperation pact signed between superpowers', 'trade_agreement'),
    ('Customs union established reducing trade barriers', 'trade_agreement'),
    ('Investment protection treaty enhances business confidence', 'trade_agreement'),
    ('Agricultural trade deal benefits exporting nations', 'trade_agreement'),
    ('Technology transfer agreement signed between allies', 'trade_agreement'),
    ('Supply chain cooperation deal strengthens ties', 'trade_agreement'),
    ('Maritime trade corridor agreement reduces shipping costs', 'trade_agreement'),
    ('Digital trade framework modernizes cross-border commerce', 'trade_agreement'),
    ('Bilateral trade pact eliminates export restrictions', 'trade_agreement'),
    ('Economic partnership reduces regulatory barriers', 'trade_agreement'),
    ('Defense trade cooperation agreement signed', 'trade_agreement'),
]

# Shuffle and split
random.seed(42)
random.shuffle(TRAINING_DATA)

split_idx = int(len(TRAINING_DATA) * 0.85)
train_data = TRAINING_DATA[:split_idx]
val_data = TRAINING_DATA[split_idx:]

print(f'Training samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print(f'Event types: {len(LABEL_MAP)}')

Training samples: 127
Validation samples: 23
Event types: 10


In [6]:
# ── Create HuggingFace Dataset ──
from datasets import Dataset

train_texts = [t[0] for t in train_data]
train_labels = [LABEL_MAP[t[1]] for t in train_data]

val_texts = [t[0] for t in val_data]
val_labels = [LABEL_MAP[t[1]] for t in val_data]

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

print(f'Train: {train_dataset}')
print(f'Val: {val_dataset}')

C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: Dataset({
    features: ['text', 'label'],
    num_rows: 127
})
Val: Dataset({
    features: ['text', 'label'],
    num_rows: 23
})


In [7]:
# Tokenize datasets
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Disable multiprocessing for Windows compatibility and small dataset
# Remove the 'text' column as the model doesn't need it, only input_ids and attention_mask
train_tokenized = train_dataset.map(tokenize_fn, batched=True).remove_columns(['text'])
val_tokenized = val_dataset.map(tokenize_fn, batched=True).remove_columns(['text'])

print('✅ Tokenization complete')

Map: 100%|██████████| 23/23 [00:00<00:00, 3776.88 examples/s]

✅ Tokenization complete


In [8]:
# Load model
from transformers import AutoModelForSequenceClassification

# Clear memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Load
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(LABEL_MAP),
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f'✅ Model ready on {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M\n')


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5723.75it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model ready on cpu
   Parameters: 67.0M



In [9]:
# Train
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro', zero_division=0),
    }

# Optimized Training config for Local GPU
args = TrainingArguments(
    output_dir=f'{MODEL_DIR}/checkpoints',
    num_train_epochs=5,               # REDUCED: 30 is insane for fine-tuning. 5 is usually enough.
    per_device_train_batch_size=16,   # Adjusted for balance
    per_device_eval_batch_size=32,    # Faster eval
    learning_rate=2e-5,               # Standard for transformers
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,               # Only keep the very best checkpoint + current
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    seed=42,
    fp16=torch.cuda.is_available(),   # Mixed precision for speed
    dataloader_num_workers=0,         # SET TO 0: Multiprocessing on Windows causes crashes/hangs.
    optim="adamw_torch"
)

# Train with Early Stopping
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if no improvement for 2 evals
)

print('🚀 Training started...\n')
trainer.train()

print('\n✅ Training complete!')

🚀 Training started...



C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,2.313966,0.043478,0.010417
2,No log,2.267349,0.043478,0.044444
3,No log,2.227928,0.217391,0.224868
4,No log,2.188240,0.304348,0.295238
5,No log,2.172027,0.347826,0.321693


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.80it/s]
C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.41it/s]
C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.38s/it]
C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\dat


✅ Training complete!


In [10]:
# Evaluate
from transformers.utils.notebook import NotebookProgressCallback

# Fix for "on_train_begin" error if it happens
if trainer.pop_callback(NotebookProgressCallback):
    print("Removed NotebookProgressCallback for evaluation stability")

print('📊 Final Results:')
results = trainer.evaluate()
print(f"  Accuracy: {results['eval_accuracy']*100:.1f}%")
print(f"  F1 Score: {results['eval_f1']*100:.1f}%")
print(f"  Loss: {results['eval_loss']:.4f}\n")

Removed NotebookProgressCallback for evaluation stability
📊 Final Results:


C:\Users\abhir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  Accuracy: 34.8%
  F1 Score: 32.2%
  Loss: 2.1720



In [11]:
# Save model locally
import os
import shutil

print('💾 Saving model...\n')

# Save model
best_model_dir = f'{MODEL_DIR}/best_model'
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

# Save label map
with open(f'{best_model_dir}/label_map.json', 'w') as f:
    json.dump({'label2id': LABEL_MAP, 'id2label': ID_TO_LABEL}, f)

print(f'✅ Model saved to {os.path.abspath(best_model_dir)}')

# Create zip for download (just for portability)
print('\n📦 Creating package...')
zip_path = './vorq_model.zip'

# Zip the model directory
shutil.make_archive('vorq_model', 'zip', best_model_dir)

# Check size
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'✅ Package ready: {os.path.abspath(zip_path)} ({size_mb:.1f} MB)')
print('\n📥 Model is saved locally on your machine.')

💾 Saving model...



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


✅ Model saved to c:\Users\abhir\OneDrive\Desktop\vorq\vorq_model_output\best_model

📦 Creating package...
✅ Package ready: c:\Users\abhir\OneDrive\Desktop\vorq\vorq_model.zip (235.8 MB)

📥 Model is saved locally on your machine.


In [14]:
# Test the model
from transformers import pipeline

print('🧪 Quick Test:\n')

classifier = pipeline('text-classification', model=best_model_dir, device=0)

tests = [
    'War between India and China',
    'US sanctions on chips',
    'Earthquake in Japan',
]

for text in tests:
    pred = classifier(text)[0]
    print(f'  "{text}" → {pred["label"]} ({pred["score"]:.1%})')

print('\n✅ Model working perfectly!')


🧪 Quick Test:



Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8366.40it/s]

  "War between India and China" → war (28.4%)
  "US sanctions on chips" → sanctions (37.2%)
  "Earthquake in Japan" → natural_disaster (44.8%)

✅ Model working perfectly!


## 🎉 Done!

Your trained model is saved locally.

### Next Steps

**Step 1:** The model is saved in `vorq_model_output/best_model`.

**Step 2:** Extract or move `best_model` folder to your VORQ project folder `vorq/models/event_classifier/`

**Step 3:** Restart VORQ

**Done!** The API now uses your trained model. 🚀